# Presenting a Fitness Studio Expansion Analysis

**Scenario.** Apex Athletic Co. has finished its sensitivity and scenario analysis.
The VP of Real Estate wants a decision brief: which studio format makes the most
sense, why, and what could make that advice wrong?

The **Given analysis** section below runs automatically — review it to understand
the numbers before building your charts. Your job starts at Chart 1.

## Given analysis (run automatically — review but do not modify)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

DATA_PATH = "data/recreation_services_cpi.csv"

OPERATING_DAYS = 350
DISCOUNT_RATE  = 0.08
LEASE_YEARS    = 5

FORMATS = {
    "Flagship": dict(daily_visits=400, avg_rev_per_visit=9.50,
                     op_margin=0.18, rent_annual=110_000, buildout=460_000),
    "Studio":   dict(daily_visits=140, avg_rev_per_visit=12.50,
                     op_margin=0.22, rent_annual=55_000,  buildout=195_000),
    "Express":  dict(daily_visits=50,  avg_rev_per_visit=16.00,
                     op_margin=0.26, rent_annual=32_000,  buildout=55_000),
}
STUDIO = FORMATS["Studio"]

cpi = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date").dropna()
cpi["yoy"] = cpi["cpi_recreation"].pct_change(12)
FEE_GROWTH = cpi[cpi["date"].dt.year >= cpi["date"].dt.year.max() - 5]["yoy"].dropna().mean()


def studio_npv(daily_visits, avg_rev_per_visit, op_margin, rent_annual, buildout,
               fee_growth=None, discount_rate=DISCOUNT_RATE,
               operating_days=OPERATING_DAYS, lease_years=LEASE_YEARS):
    if fee_growth is None:
        fee_growth = FEE_GROWTH
    annual_revenue = daily_visits * avg_rev_per_visit * operating_days
    annual_profit  = annual_revenue * op_margin - rent_annual
    pv = sum(annual_profit * (1 + fee_growth)**t / (1 + discount_rate)**t
             for t in range(1, lease_years + 1))
    return pv - buildout


SCENARIOS = {
    "Optimistic":  dict(**{**STUDIO, "daily_visits": 175,
                           "fee_growth": 0.055, "buildout": 180_000}),
    "Base":        dict(**STUDIO),
    "Pessimistic": dict(**{**STUDIO, "daily_visits": 105,
                           "fee_growth": 0.015, "buildout": 215_000}),
}

FLEX = [
    ("Daily visits (±25)",     "daily_visits",      115, 165),
    ("Rev per visit (±$1.50)", "avg_rev_per_visit", 11.00, 14.00),
    ("Op margin (±2pp)",       "op_margin",         0.20, 0.24),
    ("Annual rent (±$8K)",     "rent_annual",       63_000, 47_000),
]
CENTRAL_NPV = studio_npv(**STUDIO)
tornado_rows = []
for label, kwarg, low_val, high_val in FLEX:
    lo = studio_npv(**{**STUDIO, kwarg: low_val})
    hi = studio_npv(**{**STUDIO, kwarg: high_val})
    tornado_rows.append({"driver": label, "low_npv": lo, "high_npv": hi,
                         "range_npv": abs(hi - lo)})
tornado = pd.DataFrame(tornado_rows).sort_values("range_npv")

breakeven_visits = brentq(
    lambda v: studio_npv(v, STUDIO["avg_rev_per_visit"], STUDIO["op_margin"],
                         STUDIO["rent_annual"], STUDIO["buildout"]),
    10, 1_000
)
cushion = STUDIO["daily_visits"] - breakeven_visits

print("Given analysis complete.")
print(f"FEE_GROWTH: {FEE_GROWTH:.2%}")
print(f"Studio base NPV: ${CENTRAL_NPV:+,.0f}")
print(f"Break-even: {breakeven_visits:.0f} visits/day  (cushion: {cushion:.0f})")

---
## Chart 1 — "Which format delivers the best return?"

Build a grouped bar chart: one group per format, one bar per scenario (Optimistic /
Base / Pessimistic). Use one color per scenario. Add a data label on each bar showing
the NPV in $K (e.g. "+$157K"). Add a horizontal reference line at NPV = 0.

In [ ]:
SCENARIO_COLORS = {"Optimistic": "#2ca02c", "Base": "#1f77b4", "Pessimistic": "#d62728"}

# TODO: Build and display Chart 1.

*TODO: 1–2 sentences. What does Chart 1 tell the VP about which format to open?*

## Chart 2 — "What should we worry about most?"

Show only the **top 2 drivers** from the tornado (largest range). For each driver,
draw a horizontal bar from its low-NPV to its high-NPV value. Label the bar with
the dollar swing. Mark the central NPV with a dashed line.

In [ ]:
top2 = tornado.tail(2).reset_index(drop=True)

# TODO: Build and display Chart 2.
#       Draw a horizontal bar for each driver spanning its low-NPV to high-NPV range.
#       Label the dollar swing on each bar. Mark the central NPV with a dashed line.

*TODO: 1–2 sentences. What does Chart 2 tell the VP about where to focus due diligence?*

## Chart 3 — "How many members do we need through the door each day?"

Draw a two-zone horizontal bar: red (unprofitable) to the left of break-even,
green (profitable) to the right. Mark break-even and the base-case assumption
as vertical lines. Put the break-even label to the LEFT of its line and the
base-case label to the RIGHT so they don't overlap.

In [ ]:
lo_chart, hi_chart = 30, 250

# TODO: Build and display Chart 3.
#       Draw two colored zones: unprofitable (left of break-even) and profitable (right).
#       Mark break-even and the base-case assumption as vertical lines. Label both clearly.

*TODO: 1–2 sentences. What does Chart 3 tell the VP about the risk of the recommendation?*

---
## The memo

Fill in every bracketed field with a real number from the analysis above.

## Apex Athletic Co. — New Market Entry Format Analysis

**Recommendation:** [One sentence — which format and why]

**Key numbers:**
- Studio base-case 5-year NPV: $[X]
- Optimistic scenario: $[X] | Pessimistic: $[X]
- Break-even: [X] visits/day vs. base assumption of [X]/day

**Biggest risk:** [One sentence — top driver from tornado, quantified]

**What would change this recommendation:** [One sentence — the specific condition
that would flip the recommendation, e.g., "If daily foot traffic falls below X..."]